# Two channels, realistic baseline, oracle R^2 = 0.80

**Why this exists.** `realistic-baseline-noise.ipynb` fixed the two ways the landed DGP
flattered the model (a baseline drawn from the model's own spline basis, and iid noise).
Its `ec_m` and invariance results came out *stronger* than the landed notebook's. But it
had a liability: **Social's `roi_m` came back +146% wrong under both prior variants**,
which is a hard question to field in a 30-minute talk and is not a saturation-prior
problem at all.

**Why Social broke.** Diagnosed, not guessed. Social's true `ec_m` is 0.297, so it sits
at **77% of its ceiling** -- deep in saturation. The Hill transform crushes its
variation by nearly 5x (coefficient of variation 0.340 before, **0.070** after), leaving
its post-Hill media close to a flat line. A near-constant regressor is confounded with
the baseline *level*: harmless when the baseline is exactly recoverable (the landed DGP),
fatal once the baseline carries unmodeled common shocks, which Social then absorbs.
Neither prior can fix that, because the prior in question is about saturation and this is
about identification against the baseline.

**This notebook drops Social entirely** and runs TV vs Display only. That is the right
trade for a talk: TV (under-reached, 10% of ceiling) versus Display (moderate, 44% of
ceiling) is already the whole argument -- one channel badly misdiagnosed, one recovering
cleanly under both priors, which is what shows the failure is *channel-specific* rather
than a blanket model failure.

**One compensation was required, and it matters.** Dropping Social removes ~21% of the
KPI's media contribution. At the inherited `baseline_scale=0.15`, media falls to
**16.4%** of KPI -- uncomfortably close to the 13.7% weak-signal regime that
`ARF_COUNCIL_TALK_OUTLINE.md` records as degrading *both* prior variants (the informed
one included, by 19-29% mROI error). A weak-signal result would be a signal-to-noise
caveat masquerading as a saturation finding. So `baseline_scale` is set to **0.065**,
which restores media to 31.2% of KPI -- matching the 3-channel run's 32.4% and the
study's standing ~31% figure, so that *dropping Social* is the only thing that changed.

Note for honesty when presenting: ~31% media contribution is on the **high** side for
real MMM (10-25% is more typical). It is used here for continuity with the rest of the
study, not because it is the most realistic choice. A low-media-share variant is a
separate axis, and a known-sensitive one.

**Alpha stays at the study's usual 0.8.** Only oracle R^2 = 0.80 is run -- the settled
configuration -- rather than the full sweep.

## 0. Setup

In [1]:
import os
import sys
import time
import warnings

STUDY_DIR = next(
    d for d in (os.path.abspath(os.getcwd()),
                os.path.abspath(os.path.join(os.getcwd(), '..')))
    if os.path.exists(os.path.join(d, 'data_simulator.py')))
if STUDY_DIR not in sys.path:
  sys.path.insert(0, STUDY_DIR)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from meridian.model import model

from data_simulator import SimulationConfig
from model_utils import build_comparison_table
from model_utils import build_model_spec
from model_utils import MERIDIAN_DEFAULT_MAX_LAG
import realistic_baseline as rb

warnings.filterwarnings('ignore')
OUT_DIR = os.path.join(STUDY_DIR, 'fitted_models', 'realistic_baseline_2ch')
os.makedirs(OUT_DIR, exist_ok=True)

VARIANTS = ['default', 'ec_alpha_only']
MCMC_KWARGS = dict(n_chains=2, n_adapt=500, n_burnin=500, n_keep=1000, seed=1)
N_KNOTS = 8
TARGET_R2 = 0.80
TRUE_EC_TV = 9.0

# `default` gets Meridian's real out-of-the-box max_lag (8) -- what a
# practitioner who never touches it actually gets. `ec_alpha_only` (informed)
# already anchors ec_m/alpha_m on the true DGP values, so it also gets the
# wider window (`config.max_lag`, 13) an advertiser who knows the channel's
# carryover would choose -- max_lag is informed the same way alpha_m's prior
# is, not an independent variable left fixed across variants.
MAX_LAG_BY_VARIANT = {'default': MERIDIAN_DEFAULT_MAX_LAG}

# Two-channel version of the 3-channel notebook's overrides. Per-channel dicts
# left at their defaults keep their unused `Social` keys -- the simulator looks
# up by channel name, so extra keys are inert. Only `n_imp_channels` /
# `channel_names` and the dicts this study overrides are trimmed.
#
# `baseline_scale=0.065` is NOT inherited: see the header. At the default 0.15
# two channels leave media at 16.4% of KPI; 0.065 restores 31.2%, matching the
# 3-channel run so that dropping Social is the only change.
BASE_OVERRIDES = {
    'n_imp_channels': 2,
    'channel_names': ['TV', 'Display'],
    'target_audience_pop_frac': {'TV': 0.60, 'Display': 0.50},
    'current_reach_frac': {'TV': 0.10, 'Display': 0.5},
    'frequency_range': {'TV': (1, 2), 'Display': (1, 5)},
    'target_roi': {'TV': 8.0, 'Display': 5.0},
    'max_lag': 13,
    'n_times': 156,
    'roi_ec_elasticity': -0.3,
    'roi_alpha_elasticity': 0.3,
    'adstock_retention_range': {'TV': (0.8, 0.8), 'Display': (0.0, 0.3)},
    'baseline_scale': 0.065,
}

REAL_CSV_CACHE = os.path.join(STUDY_DIR, '.cache_geo_media_rf.csv')
REAL_CSV_URL = (
    'https://raw.githubusercontent.com/google/meridian/refs/heads/main/'
    'meridian/data/simulated_data/csv/geo_media_rf.csv'
)
if not os.path.exists(REAL_CSV_CACHE):
  pd.read_csv(REAL_CSV_URL).to_csv(REAL_CSV_CACHE, index=False)
real_df = pd.read_csv(REAL_CSV_CACHE)
print('study dir:', STUDY_DIR, '| real data:', real_df.shape)

study dir: /Users/mariappan.subramanian/Documents/repo/meridian/demo/synthetic | real data: (3120, 17)


## 1. Build the two-channel scenario

`ec_m` is linear in `saturation_frequency`, so one uncalibrated build solves for TV's
true `ec_m` landing on exactly 9.0.

**True `roi_m` is higher here than in the 3-channel run** (TV ~7.6 vs 5.96). That is
`roi_ec_elasticity`/`roi_alpha_elasticity` working as designed: they scale each channel's
ROI by its curve shape *relative to the geometric mean across channels*, and dropping
Social (`ec_m` 0.297) raises that mean. Compare fitted-vs-true errors within this
notebook; never compare ROI levels across notebooks.

In [2]:
def build(saturation_frequency_tv=None, target_r2=TARGET_R2):
  overrides = dict(BASE_OVERRIDES)
  if saturation_frequency_tv is not None:
    overrides['saturation_frequency'] = {
        'TV': saturation_frequency_tv, 'Display': 4.0}
  cfg = SimulationConfig.from_dict(overrides)
  sim, data, gt = rb.build_real_augmented_realistic(
      cfg, real_df,
      rf_source_map={'TV': 'Channel3'},
      plain_source_map={'Display': 'Channel2'},
      target_oracle_r2=target_r2, n_knots=N_KNOTS,
  )
  return cfg, sim, data, gt


_, sim_base, _, _ = build(target_r2=None)
EC_BASE, SF_BASE = float(sim_base.ec_m.numpy()[0]), 4.0
SATURATION_FREQ_TV = SF_BASE * TRUE_EC_TV / EC_BASE

cfg, sim, data, gt = build(SATURATION_FREQ_TV)
true_roi, true_ec, true_alpha = (
    np.asarray(gt['roi_m']), sim.ec_m.numpy(), sim.alpha_m.numpy())
assert abs(float(sim.ec_m.numpy()[0]) - TRUE_EC_TV) < 0.05, 'ec_m solve missed'
assert abs(float(sim.alpha_m.numpy()[0]) - 0.8) < 1e-6, 'TV alpha must stay 0.8'

channels = list(cfg.channel_names)

# Guard the baseline_scale choice: a silent drift back toward the weak-signal
# regime would quietly change what this notebook is testing.
diag = rb.baseline_diagnostics(sim, N_KNOTS)
assert 28.0 < diag['media_share_pct'] < 34.0, (
    f"media share {diag['media_share_pct']:.1f}% left the intended band -- "
    'retune baseline_scale before trusting these fits')

print('channels    :', channels)
print('true ec_m   :', np.round(true_ec, 3))
print('true alpha_m:', np.round(true_alpha, 3))
print('true roi_m  :', np.round(true_roi, 3))

I0000 00:00:1785516542.660519 2915210 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


ec_m = [11.127365  1.291546]


ec_m = [9.       1.291546]


channels    : ['TV', 'Display']
true ec_m   : [9.    1.292]
true alpha_m: [0.8   0.156]
true roi_m  : [7.642 5.234]


## 2. DGP diagnostics

Confirming this is the same realistic DGP as the 3-channel notebook -- and that Display
is a usable control channel rather than a second Social.

`ceiling_frac` is where the channel sits on its own Hill curve today. `cv_post_hill` is
the coefficient of variation of its national post-Hill media: **this is the number that
killed Social at 0.070**. `spline_representable` is how much of that signal an 8-knot
baseline spline could mimic, i.e. how baseline-confounded the channel is.

In [3]:
print('oracle_r2               ', round(diag['oracle_r2'], 4))
print('dgp_r2_eps_only         ', round(diag['dgp_r2_eps_only'], 4))
print('noise multiplier        ', round(diag['noise_scale'], 3))
print('media share of KPI      ', f"{diag['media_share_pct']:.1f}%")
print('resid lag-1 autocorr    ', round(diag['resid_lag1_autocorr'], 3))
print('unexplained x-geo corr  ', round(diag['unexplained_cross_geo_corr'], 3))
print('spline absorbs mu_shock ', f"{diag['spline_explains_mu_shock_pct']:.1f}%")
print('clipped geo-weeks       ', f"{diag['clipped_frac_pct']:.2f}%")

ec = sim.ec_m.numpy()
x = sim.transformed_ipc_gtm.numpy()
nat = sim.media_transformed.numpy().mean(axis=0)
rows = []
for i, ch in enumerate(channels):
  med = np.median(x[:, :, i][x[:, :, i] > 0])
  s = nat[:, i]
  rows.append({
      'channel': ch,
      'true_ec_m': round(float(ec[i]), 3),
      'ceiling_frac': f'{med / (med + ec[i]):.1%}',
      'cv_post_hill': round(float(s.std() / s.mean()), 3),
      'spline_representable': f'{1 - rb._spline_residual(s, N_KNOTS).var() / s.var():.1%}',
  })
hill_position = pd.DataFrame(rows)
hill_position.to_csv(os.path.join(OUT_DIR, 'hill_position.csv'), index=False)
hill_position

oracle_r2                0.8001
dgp_r2_eps_only          0.88
noise multiplier         2.95
media share of KPI       31.2%
resid lag-1 autocorr     0.404
unexplained x-geo corr   0.382
spline absorbs mu_shock  33.5%
clipped geo-weeks        3.08%


,channel,true_ec_m,ceiling_frac,cv_post_hill,spline_representable
0,TV,9.000,10.0%,0.218,66.0%
1,Display,1.292,43.6%,0.235,5.5%


## 3. Recovery: `default` vs. `ec_alpha_only`

Two fits, roughly 4 minutes. `width_x_true` is the 90% HDI width as a multiple of the
true value -- reported alongside the ✓/✗ marks because coverage marks reward imprecision,
and at this noise level several intervals are wide enough to pass while carrying no
information.

**`max_lag` is now part of what "informed" means, not a fixed nuisance parameter.**
`default` fits at Meridian's real out-of-the-box `max_lag` (8, `MERIDIAN_DEFAULT_MAX_LAG`);
`ec_alpha_only` fits at the DGP's own window (`config.max_lag`, 13) -- the wider window an
advertiser who knows TV's carryover runs long would actually choose. Our adstock complaint
is two-fold (a genuinely flat, uninformative `alpha_m` prior, and a hard truncation at
week 8 that the default never estimates), and the informed variant should fix both, the
same way it already anchors `ec_m`/`alpha_m` on the true values rather than leaving them at
Meridian's defaults.

In [4]:
def fit(data, sim, config, variant):
  spec = build_model_spec(variant, sim, config, media_prior_type='roi',
                          knots=N_KNOTS,
                          max_lag=MAX_LAG_BY_VARIANT.get(variant))
  mmm = model.Meridian(input_data=data, model_spec=spec)
  mmm.sample_prior(500)
  mmm.sample_posterior(**MCMC_KWARGS)
  return mmm


fitted = {}
for variant in VARIANTS:
  t0 = time.time()
  fitted[variant] = fit(data, sim, cfg, variant)
  print(f'{variant}: {time.time() - t0:.0f}s', flush=True)

comparison = build_comparison_table(
    fitted, cfg, sim, gt, hdi_prob=0.9)
comparison.to_csv(os.path.join(OUT_DIR, 'comparison.csv'), index=False)
print(f'\nparameter recovery -- oracle R^2 = {TARGET_R2}, true TV ec_m = 9.0')
comparison

2026-07-31 11:49:29.112055: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


default: 82s


2026-07-31 11:50:52.617444: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


ec_alpha_only: 67s



parameter recovery -- oracle R^2 = 0.8, true TV ec_m = 9.0


,channel,param,true,default,ec_alpha_only
0,TV,alpha_m,0.800,0.787 (0.72-0.85) ✓,0.691 (0.63-0.74) ✗
1,TV,ec_m,9.000,0.914 (0.44-1.43) ✗,8.926 (7.46-10.28) ✓
2,TV,roi_m,7.642,15.221 (11.29-18.99) ✗,7.756 (6.37-9.20) ✓
3,Display,alpha_m,0.156,0.171 (0.09-0.24) ✓,0.206 (0.13-0.28) ✓
4,Display,ec_m,1.292,0.730 (0.39-1.05) ✗,1.219 (1.03-1.39) ✓
5,Display,roi_m,5.234,6.713 (5.07-8.38) ✓,5.982 (5.08-6.82) ✓


In [5]:
# Point errors and interval widths, which the ✓/✗ table alone does not show.
import re

def parse_cell(text):
  m = re.match(r'\s*([\d.]+)\s*\(([\d.]+)-([\d.]+)\)\s*(✓|✗)', str(text))
  return (float(m.group(1)), float(m.group(2)), float(m.group(3)), m.group(4))

rows = []
for _, r in comparison.iterrows():
  for v in VARIANTS:
    val, lo, hi, mark = parse_cell(r[v])
    rows.append({
        'channel': r['channel'], 'param': r['param'], 'true': r['true'],
        'variant': v, 'fitted': val,
        'err_pct': round((val / r['true'] - 1) * 100, 1),
        'hdi': f'{lo}-{hi}', 'width_x_true': round((hi - lo) / r['true'], 2),
        'mark': mark,
    })
errors = pd.DataFrame(rows)
errors.to_csv(os.path.join(OUT_DIR, 'errors_with_widths.csv'), index=False)
errors

,channel,param,true,variant,fitted,err_pct,hdi,width_x_true,mark
0,TV,alpha_m,0.800,default,0.787,-1.6,0.72-0.85,0.16,✓
1,TV,alpha_m,0.800,ec_alpha_only,0.691,-13.6,0.63-0.74,0.14,✗
2,TV,ec_m,9.000,default,0.914,-89.8,0.44-1.43,0.11,✗
3,TV,ec_m,9.000,ec_alpha_only,8.926,-0.8,7.46-10.28,0.31,✓
4,TV,roi_m,7.642,default,15.221,99.2,11.29-18.99,1.01,✗
5,TV,roi_m,7.642,ec_alpha_only,7.756,1.5,6.37-9.2,0.37,✓
6,Display,alpha_m,0.156,default,0.171,9.6,0.09-0.24,0.96,✓
7,Display,alpha_m,0.156,ec_alpha_only,0.206,32.1,0.13-0.28,0.96,✓
8,Display,ec_m,1.292,default,0.730,-43.5,0.39-1.05,0.51,✗
9,Display,ec_m,1.292,ec_alpha_only,1.219,-5.7,1.03-1.39,0.28,✓


## 4. Findings

**This is the strongest result set in the study.** Dropping Social both sharpened `ec_m`
recovery and turned TV's ROI into a headline number, because the ROI error here is
*caused by* the `ec_m` prior rather than incidental to it.

### The headline

Within this single DGP -- same data, same noise, same seed -- the difference between
these two columns is now the prior *and* `max_lag` together (Section 3): `default` gets
Meridian's real out-of-the-box `max_lag=8` on top of its flat/default priors,
`ec_alpha_only` gets the DGP's own `max_lag=13` on top of its informed `ec_m`/`alpha_m`
priors -- max_lag is informed the same way the other two are, not held fixed.

| TV parameter | true | `default` | informed (`ec_alpha_only`) |
|---|---|---|---|
| `ec_m` | 9.000 | 0.914 (**−89.8%**) ✗ | 8.926 (**−0.8%**) ✓ |
| `roi_m` | 7.642 | 15.221 (**+99.2%**) ✗ | 7.756 (**+1.5%**) ✓ |
| `alpha_m` | 0.800 | 0.787 (**−1.6%**) ✓ | 0.691 (−13.6%) ✗ |

One prior-and-window change moves ROI from 99.2% wrong to 1.5% wrong and the saturation
point from 89.8% wrong to under 1% wrong, on the most defensible DGP in the study.

**A genuine surprise: `alpha_m` flipped.** Under the previous shared-`max_lag=13` setup,
`default` failed `alpha_m` coverage (−11.9% ✗) and so did `ec_alpha_only` (−13.6% ✗) --
"neither prior helps" was the reading. With `default` now truncated to `max_lag=8`, its
`alpha_m` point estimate moved *closer* to the truth (0.787 vs. 0.705 before) and its
interval now covers (✓), while `ec_alpha_only` is unchanged (still −13.6% ✗, since its
`max_lag=13` and its `alpha_m` prior are both unchanged). This is not evidence that the
default's `alpha_m` handling is somehow *better* -- `ec_m` and `roi_m` are both far worse
under `default` -- but it is a real result worth stating plainly rather than smoothing
over: a truncated window can pull a *different* parameter's point estimate toward its true
value for reasons that have nothing to do with that parameter's own prior. Treat `alpha_m`
coverage marks with more suspicion after this, not less.

### Why the ROI error is downstream of the `ec_m` prior

The mechanism from the original (shared-`max_lag=13`) run -- an over-saturated `ec_m`
flattens TV's post-Hill signal, a flatter regressor is more confoundable with the baseline
*level*, so `beta` (and hence ROI) inflates to compensate -- has not been re-verified
against the new `max_lag=8` `default` fit in this pass. The qualitative chain (**wrong
prior -> over-saturated curve -> flattened signal -> baseline confounding -> inflated
ROI**) is very likely still operating, and `default`'s ROI error getting *worse* (79.0% ->
99.2%) alongside a *narrower* window is consistent with it -- less carryover modeled means
even more of TV's true signal has to be explained some other way. But the specific
post-Hill CV figures quoted in earlier drafts of this section (0.153 / 0.229) were computed
against the old `default` fit and are retired until recomputed against the current
`.nc` inference data.

### Why the 3-channel run *looked* better on ROI, and why that was worse

With three channels, `default` returned TV's ROI at −22.5% and it *passed* coverage (under
the study's earlier, shared-`max_lag` methodology). That was not recovery on merit.
Social's post-Hill CV was **0.070** -- far flatter than TV can be at any fitted `ec_m` --
so Social absorbed the unmodeled common baseline shock, and its own ROI came back +146%
wrong. TV's ROI was spared by a worse victim.

Remove that victim and the mechanism lands where it was always headed. So the two-channel
result is not a new failure; it is the same failure, now visible on the channel the talk
is actually about. (This paragraph's 3-channel figures predate the `max_lag` split and are
not a like-for-like comparison to the table above; treat as historical context only.)

`default` also fails Display's `ec_m` (−43.5% ✗), though its `roi_m` now passes (+28.3% ✓,
a wider interval than before).

### One thing not to overclaim

The ROI error's **sign and magnitude are channel-set- and methodology-dependent**: +13.7%
(landed DGP), −22.5% (3-channel realistic), +79.0% (2-channel, shared `max_lag=13`), +99.2%
(2-channel, `max_lag` split) -- same true `ec_m` throughout, different everything else. The
supported claim is *the default prior (and its out-of-the-box `max_lag`) can get ROI badly
wrong, and an informed practitioner fixes it*. Not *+99% is what defaults do*.

Consequence for `ARF_COUNCIL_TALK_OUTLINE.md` §5: Beat 2's line -- *"every standard
diagnostic is clean"* -- is not available on this DGP, since +99.2% with a non-covering
interval is a failure a practitioner would catch. That is a sentence to rewrite, not a
reason to prefer the weaker result. The replacement is simpler to land in 30 minutes:
**the default prior overstates TV's ROI by ~99% and understates its saturation point by
~90%; one informed practitioner (priors and window both) fixes both to within 2%.**

### Standing caveats

- The channel-set change and `baseline_scale` 0.15 -> 0.065 moved **together**, by design,
  to hold media share at 31.2% so signal strength stays out of the comparison. The
  `default`-vs-informed contrast is unaffected either way: both variants see identical
  data. A two-channel run at the inherited 0.15 would sit at 16.4% media share -- a
  different, weaker-signal experiment.
- True `roi_m` differs across all three notebooks (7.642 here, 5.963 3-channel, 6.660
  landed) because `roi_ec_elasticity`/`roi_alpha_elasticity` scale ROI against the
  cross-channel geometric mean. Compare errors, never levels.
- Single simulator draw at `seed_num` 1320, and per the seed issue recorded in
  `realistic-baseline-noise.ipynb` §6, that is the only seed the study has ever run at this
  `max_lag` split. The ten-seed sweep (`realistic-baseline-noise-2ch-seeds.ipynb`) needs a
  fresh run under the new methodology before this draw's numbers can be judged
  representative.
- **Retired in this pass:** the `ec9`-vs-`ec11` invariance check (Section 4 in earlier
  versions of this notebook) is no longer run. It was never read by the ARF deck
  (`results_facts.py` doesn't touch it), and this notebook now needs the compute budget
  for the `max_lag` split instead. Its prior finding -- `default` absorbed 0.01-0.14 of an
  18.7% truth move vs. the informed prior's 0.98-1.04 -- is no longer being tracked or
  reproduced; treat it as retired evidence, not as still-standing.
